# Testing Clif -- growing the Clifford engine
<!-- 
2026.05.04 DAJones: created
2026.05.06 DAJones: working shirokov inverse
-->
* Working in the Engine directory.
* Clif.py - user interface, initialization.
* SignTable6.py - sign tables, signature and degeneracy support, optimized multiplication is done here.
* Accum.py - the accumulator - implementing a multivector

<pre>
    These Benchmarks are with a jitted multivector multiply only.
    
    Benchmark for 6D Jones Inverse: average time = 116.60480231512338 microseconds
    Benchmark for 6D Shirokov Inverse: average time = 171.27730243373662 microseconds
    
    Benchmark for 7D Shirokov Inverse: average time = 721.0714014945552 microseconds
</pre>

In [1]:
# this cell just verifies working algorithms

import util
from shirokov_inverse import *

A = util.random( signature = 0, dimensions = 6 )
print( "Type of A.Reg:", type(A.Reg) ) # want numpy array, not list
Ij = util.I6(A)
print( '\nJones inverse at work. Ij*A = \n', Ij*A )

Is = shirokov_inverse( A )
print( '\nShirokov inverse at work. Is*A = \n', Is*A )
util.print2Accum( Ij, Is )

Type of A.Reg: <class 'numpy.ndarray'>

Jones inverse at work. Ij*A = 
 00000000       1.00000000


Shirokov inverse at work. Is*A = 
 00000000       1.00000000

  0. 00000000     -0.06959457     -0.06959457
  1. 00000001     -0.03710588     -0.03710588
  2. 00000010     -0.07268145     -0.07268145
  3. 00000011      0.00229709      0.00229709
  4. 00000100      0.03764348      0.03764348
  5. 00000101     -0.04932597     -0.04932597
  6. 00000110      0.01638573      0.01638573
  7. 00000111     -0.02048731     -0.02048731
  8. 00001000      0.00324173      0.00324173
  9. 00001001      0.00908314      0.00908314
 10. 00001010     -0.01056336     -0.01056336
 11. 00001011     -0.03693343     -0.03693343
 12. 00001100     -0.05596938     -0.05596938
 13. 00001101     -0.03242881     -0.03242881
 14. 00001110     -0.03296988     -0.03296988
 15. 00001111     -0.08841702     -0.08841702
 16. 00010000      0.06673143      0.06673143
 17. 00010001      0.02408134      0.02408134
 18. 00010

In [2]:
import time
import util
from shirokov_inverse import *

# Generate a test multivector 
A = util.random(signature=0, dimensions=6)

# ensure util.random outputs a NumPy array!
print("multivectors are type = ",type(A.Reg))

def TimeIt( iterations, Inverse, A ):
    I = Inverse(A) # a throw away first time
    total_time = 0
    for _ in range(iterations):
        start_time = time.perf_counter()
        I = Inverse(A)
        end_time = time.perf_counter()
        elapsed_time = end_time - start_time
        print(f"{elapsed_time * 1e6:.2f} microseconds")
        total_time += elapsed_time
    average_usec = (total_time / iterations) * 1e6
    print(f"average time = {average_usec} microseconds")
    return average_usec

print( "\nTiming Shirokov Inverse" )
s_usec = TimeIt( 10, shirokov_inverse, A )

print( "\nTiming Jones Inverse" )
j_usec = TimeIt( 10, util.I6, A )

n = 6 # dimensions
N = 1 << ( (n + 1) // 2 )
s_mul = N-1
j_mul = 5

print()
print( f"Shirokov inverse performs {s_mul} multivector multiplications" )
print( f"Jones inverse performs {j_mul} multivector multiplications" )
print( f"Relative extra effort Shirokov/Jones = {s_mul/j_mul}" )
print( f"Relative average time Shirokov/Jones = {s_usec/j_usec}" )



multivectors are type =  <class 'numpy.ndarray'>

Timing Shirokov Inverse
166.22 microseconds
178.91 microseconds
170.43 microseconds
169.63 microseconds
183.96 microseconds
169.63 microseconds
172.62 microseconds
167.45 microseconds
166.87 microseconds
167.05 microseconds
average time = 171.27730243373662 microseconds

Timing Jones Inverse
108.29 microseconds
129.50 microseconds
114.41 microseconds
113.93 microseconds
113.81 microseconds
127.27 microseconds
113.85 microseconds
113.90 microseconds
112.81 microseconds
118.25 microseconds
average time = 116.60480231512338 microseconds

Shirokov inverse performs 7 multivector multiplications
Jones inverse performs 5 multivector multiplications
Relative extra effort Shirokov/Jones = 1.4
Relative average time Shirokov/Jones = 1.4688700553761185


In [4]:


A = util.random(signature=0, dimensions=7)
I = shirokov_inverse( A )
print("\nVerifying Shirokov Inverse for 7D")
print(A*I)

print( "\nTiming Shirokov Inverse for 7D" )
s_usec = TimeIt( 10, shirokov_inverse, A )




Verifying Shirokov Inverse for 7D
00000000       1.00000000


Timing Shirokov Inverse for 7D
696.16 microseconds
708.27 microseconds
705.44 microseconds
695.15 microseconds
716.28 microseconds
736.47 microseconds
733.13 microseconds
740.96 microseconds
733.58 microseconds
745.28 microseconds
average time = 721.0714014945552 microseconds
